## Evolution of embeddings

#### Bag of words

In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# --- MY OWN DOCUMENTS ---
docs = [
    "The cat sat on the mat and the cat purred loudly.",
    "The dog sat on the rug and barked at the cat.",
    "A bird flew over the mat while the cat watched silently.",
    "The dog chased the bird around the garden fence.",
    "A cat and a dog can be good friends despite their differences.",
    "The garden had a mat near the fence where the bird nested."
]

# --- STEP 1: PREPROCESSING ---
def simple_preprocess(text: str) -> str:
    text = text.lower()                          # "The Cat" → "the cat"
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)  # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()     # collapse whitespace
    return text

docs_pp = [simple_preprocess(d) for d in docs]
# Let's see what preprocessing did:
for i, (orig, clean) in enumerate(zip(docs, docs_pp)):
    print(f"D{i+1} original : {orig}")
    print(f"D{i+1} cleaned  : {clean}\n")

# --- STEP 2: BAG OF WORDS (unigrams only first, to keep it clear) ---
bow_uni = CountVectorizer(
    stop_words="english",   # removes "the", "on", "and", "a", "can", "be", etc.
    ngram_range=(1, 1),      # unigrams only
    min_df=1
)
X_uni = bow_uni.fit_transform(docs_pp)
vocab_uni = bow_uni.get_feature_names_out()

print("=== UNIGRAM BAG OF WORDS ===")
print(f"Shape: {X_uni.shape}  →  {X_uni.shape[0]} docs × {X_uni.shape[1]} unique terms\n")

# Show the full matrix as a readable DataFrame
df_uni = pd.DataFrame(
    X_uni.toarray(),
    columns=vocab_uni,
    index=[f"D{i+1}" for i in range(len(docs))]
)
print(df_uni)
# Each cell = how many times that term appears in that doc.
# D1 has "cat" twice → cell value is 2.

# --- STEP 3: NOW ADD BIGRAMS ---
bow_bi = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),      # unigrams + bigrams
    min_df=1
)
X_bi = bow_bi.fit_transform(docs_pp)
vocab_bi = bow_bi.get_feature_names_out()

print("\n=== UNIGRAM + BIGRAM BAG OF WORDS ===")
print(f"Shape: {X_bi.shape}  →  vocabulary jumped from {len(vocab_uni)} to {len(vocab_bi)} terms\n")

# Show only non-zero entries per doc (much easier to read)
for i in range(len(docs)):
    row = X_bi[i].toarray().flatten()
    nonzero = [(vocab_bi[j], row[j]) for j in row.nonzero()[0]]
    nonzero.sort(key=lambda x: -x[1])
    print(f"D{i+1}: {nonzero}")

# --- STEP 4: WHY "BAG"? ORDER IS LOST ---
print("\n=== WHY IT'S CALLED A 'BAG' ===")
test = ["dog bites man", "man bites dog"]
test_bow = CountVectorizer(ngram_range=(1,1)).fit_transform(test)
test_vocab = CountVectorizer(ngram_range=(1,1)).fit(test).get_feature_names_out()
df_test = pd.DataFrame(test_bow.toarray(), columns=test_vocab, index=test)
print(df_test)
print("↑ Identical rows! BoW cannot tell these apart with unigrams.")
print("  Bigrams partially fix this: 'dog bites' ≠ 'man bites'")

D1 original : The cat sat on the mat and the cat purred loudly.
D1 cleaned  : the cat sat on the mat and the cat purred loudly

D2 original : The dog sat on the rug and barked at the cat.
D2 cleaned  : the dog sat on the rug and barked at the cat

D3 original : A bird flew over the mat while the cat watched silently.
D3 cleaned  : a bird flew over the mat while the cat watched silently

D4 original : The dog chased the bird around the garden fence.
D4 cleaned  : the dog chased the bird around the garden fence

D5 original : A cat and a dog can be good friends despite their differences.
D5 cleaned  : a cat and a dog can be good friends despite their differences

D6 original : The garden had a mat near the fence where the bird nested.
D6 cleaned  : the garden had a mat near the fence where the bird nested

=== UNIGRAM BAG OF WORDS ===
Shape: (6, 21)  →  6 docs × 21 unique terms

    barked  bird  cat  chased  despite  differences  dog  fence  flew  \
D1       0     0    2       0        

In [7]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- SAME DOCUMENTS ---
docs = [
    "The cat sat on the mat and the cat purred loudly.",
    "The dog sat on the rug and barked at the cat.",
    "A bird flew over the mat while the cat watched silently.",
    "The dog chased the bird around the garden fence.",
    "A cat and a dog can be good friends despite their differences.",
    "The garden had a mat near the fence where the bird nested."
]

def simple_preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

docs_pp = [simple_preprocess(d) for d in docs]

# =============================================================
# --- STEP 1: RAW BOW COUNTS (for comparison) ---
# =============================================================
bow = CountVectorizer(stop_words="english", ngram_range=(1,1), min_df=1)
X_bow = bow.fit_transform(docs_pp)
vocab = bow.get_feature_names_out()

print("=== RAW BAG OF WORDS (just counts) ===\n")
df_bow = pd.DataFrame(X_bow.toarray(), columns=vocab, index=[f"D{i+1}" for i in range(len(docs))])
print(df_bow)

# =============================================================
# --- STEP 2: TF-IDF ---
# =============================================================
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 1),
    norm="l2",            # normalize each doc vector to unit length
    use_idf=True,         # enable inverse document frequency
    smooth_idf=True,      # add 1 to df to prevent division by zero
    sublinear_tf=False    # keep TF linear first (we'll compare later)
)
X_tfidf = tfidf.fit_transform(docs_pp)
vocab_tfidf = tfidf.get_feature_names_out()

print("\n=== TF-IDF MATRIX ===\n")
df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=vocab_tfidf,
    index=[f"D{i+1}" for i in range(len(docs))]
)
print(df_tfidf)

# =============================================================
# --- STEP 3: UNDERSTAND THE IDF VALUES ---
# =============================================================
print("\n=== IDF VALUES (how rare/informative each word is) ===\n")

# idf formula: log((n_docs + 1) / (df + 1)) + 1  (with smooth_idf=True)
# n_docs = 6, so:
#   word in ALL 6 docs → idf = log(7/7) + 1 = 1.0   (lowest weight)
#   word in only 1 doc → idf = log(7/2) + 1 = 2.25   (highest weight)

idf_df = pd.DataFrame({
    "word": vocab_tfidf,
    "idf": tfidf.idf_.round(3),
    "appears_in_n_docs": [int((X_bow[:, i].toarray() > 0).sum()) for i in range(len(vocab_tfidf))]
}).sort_values("idf", ascending=True)

print(idf_df.to_string(index=False))
print("""
↑ Words appearing in many docs get LOW idf → they're common, less informative.
  Words appearing in few docs get HIGH idf → they're rare, more informative.
  
  'cat' appears in 4/6 docs → idf ≈ 1.22  (common, downweighted)
  'purred' appears in 1/6 docs → idf ≈ 2.25  (rare, boosted!)
""")

# =============================================================
# --- STEP 4: SIDE BY SIDE — BoW vs TF-IDF for ONE document ---
# =============================================================
print("=== D1 SIDE BY SIDE: BoW counts vs TF-IDF weights ===\n")
print(f'D1: "{docs[0]}"\n')

d1_bow = X_bow[0].toarray().flatten()
d1_tfidf = X_tfidf[0].toarray().flatten()

comparison = pd.DataFrame({
    "word": vocab,
    "bow_count": d1_bow.astype(int),
    "tfidf_weight": d1_tfidf.round(3)
}).query("bow_count > 0").sort_values("tfidf_weight", ascending=False)

print(comparison.to_string(index=False))
print("""
↑ In BoW, 'cat' has count=2 (highest raw count in D1).
  In TF-IDF, 'cat' is DOWNWEIGHTED because it appears in 4/6 docs.
  'purred' and 'loudly' get BOOSTED — they're unique to D1!
  
  TF-IDF says: "don't just count words, count INFORMATIVE words."
""")

# =============================================================
# --- STEP 5: DOCUMENT SIMILARITY — BoW vs TF-IDF ---
# =============================================================
print("=== DOCUMENT SIMILARITY (cosine) ===\n")

sim_bow = cosine_similarity(X_bow)
sim_tfidf = cosine_similarity(X_tfidf)

print("Using raw BoW counts:")
df_sim_bow = pd.DataFrame(
    sim_bow.round(3),
    index=[f"D{i+1}" for i in range(len(docs))],
    columns=[f"D{i+1}" for i in range(len(docs))]
)
print(df_sim_bow)

print("\nUsing TF-IDF weights:")
df_sim_tfidf = pd.DataFrame(
    sim_tfidf.round(3),
    index=[f"D{i+1}" for i in range(len(docs))],
    columns=[f"D{i+1}" for i in range(len(docs))]
)
print(df_sim_tfidf)
print("""
↑ Compare the two matrices:
  TF-IDF usually gives LOWER similarities because it downweights
  shared common words (like 'cat') that inflate BoW similarity.
  The similarities that REMAIN in TF-IDF are more meaningful —
  they're driven by shared informative words, not shared filler.
""")

# =============================================================
# --- STEP 6: SUBLINEAR TF (log scaling) ---
# =============================================================
print("=== BONUS: sublinear_tf=True (logarithmic term frequency) ===\n")

tfidf_sub = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 1),
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True     # tf = 1 + log(tf) instead of raw count
)
X_tfidf_sub = tfidf_sub.fit_transform(docs_pp)

d1_sub = X_tfidf_sub[0].toarray().flatten()

compare_tf = pd.DataFrame({
    "word": vocab,
    "raw_count": d1_bow.astype(int),
    "linear_tfidf": d1_tfidf.round(4),
    "sublinear_tfidf": d1_sub.round(4)
}).query("raw_count > 0").sort_values("raw_count", ascending=False)

print(f'D1: "{docs[0]}"\n')
print(compare_tf.to_string(index=False))
print("""
↑ With sublinear_tf=True:
  'cat' (count=2) gets tf = 1 + log(2) = 1.69 instead of 2.
  'purred' (count=1) gets tf = 1 + log(1) = 1.0 (same).
  
  This squashes the advantage of repeated words.
  Appearing 10 times isn't 10x more important than appearing once —
  sublinear TF captures this intuition with logarithmic scaling.
""")

=== RAW BAG OF WORDS (just counts) ===

    barked  bird  cat  chased  despite  differences  dog  fence  flew  \
D1       0     0    2       0        0            0    0      0     0   
D2       1     0    1       0        0            0    1      0     0   
D3       0     1    1       0        0            0    0      0     1   
D4       0     1    0       1        0            0    1      1     0   
D5       0     0    1       0        1            1    1      0     0   
D6       0     1    0       0        0            0    0      1     0   

    friends  ...  good  loudly  mat  near  nested  purred  rug  sat  silently  \
D1        0  ...     0       1    1     0       0       1    0    1         0   
D2        0  ...     0       0    0     0       0       0    1    1         0   
D3        0  ...     0       0    1     0       0       0    0    0         1   
D4        0  ...     0       0    0     0       0       0    0    0         0   
D5        1  ...     1       0    0     0  

#### Word2Vec

In [10]:
pip install gensim==4.3.3 scipy==1.12.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 7.8 MB/s  0:00:03m 7.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 8.4 MB/s  0:00:038.4 MB/s eta 0:00:01:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1
  Attempting uninstall: gensim━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [scipy]
    Found existing installation: gensim 4.3.2━━━━━ 0/2 [scipy]
    Uninstalling gensim-4.3.2:━━━━━━━━━━━━━━━━━━━━ 0/2 [scipy]
      Successfully uninstalled gensim-4.3.2━━━━━━━ 0/2 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━ 1/2 [gensim]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.1.72 requires flatbuffers<3.0,>=1.12, but you have flatbuffers 23.5

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

# --- SAME DOCUMENTS ---
docs = [
    "The cat sat on the mat and the cat purred loudly.",
    "The dog sat on the rug and barked at the cat.",
    "A bird flew over the mat while the cat watched silently.",
    "The dog chased the bird around the garden fence.",
    "A cat and a dog can be good friends despite their differences.",
    "The garden had a mat near the fence where the bird nested."
]

def simple_preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- TOKENIZE INTO WORD LISTS (Word2Vec needs list of lists) ---
docs_tokenized = [simple_preprocess(d).split() for d in docs]

print("=== TOKENIZED DOCUMENTS ===\n")
for i, tokens in enumerate(docs_tokenized):
    print(f"D{i+1}: {tokens}\n")

# =============================================================
# --- STEP 1: TRAIN WORD2VEC ---
# =============================================================
model = Word2Vec(
    sentences=docs_tokenized,
    vector_size=10,       # embedding dimensions (small for demo; real = 100-300)
    window=3,             # context window: look 3 words left and right
    min_count=1,          # keep all words (even if they appear once)
    sg=1,                 # 1 = Skip-gram, 0 = CBOW
    epochs=200,           # more epochs for tiny corpus
    seed=42
)

vocab_words = list(model.wv.key_to_index.keys())
print(f"=== WORD2VEC MODEL TRAINED ===")
print(f"Vocabulary: {len(vocab_words)} words")
print(f"Embedding dimensions: {model.wv.vector_size}\n")

# =============================================================
# --- STEP 2: SEE THE ACTUAL EMBEDDINGS ---
# =============================================================
print("=== WORD EMBEDDINGS (each word is now a dense vector) ===\n")

# Compare to BoW one-hot: was [0,0,1,0,0,...0] (20+ dims, one 1)
# Now: [0.23, -0.11, 0.87, ...] (10 dims, ALL meaningful)

key_words = ["cat", "dog", "bird", "mat", "garden", "fence", "sat", "chased"]
key_words = [w for w in key_words if w in model.wv]  # keep only words in vocab

for word in key_words:
    vec = model.wv[word]
    print(f"'{word:10s}' → {np.array2string(vec, precision=3, suppress_small=True)}")

print(f"""
↑ COMPARE TO BOW ONE-HOT:
  BoW 'cat'  = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]  (sparse, meaningless)
  W2V 'cat'  = {np.array2string(model.wv['cat'], precision=2)}  (dense, learned)

  Every dimension now carries information.
  Similar words get pushed toward similar vectors during training.
""")

# =============================================================
# --- STEP 3: WORD SIMILARITY — THE BIG DIFFERENCE ---
# =============================================================
print("=== WORD-TO-WORD COSINE SIMILARITY ===\n")

embeddings = np.array([model.wv[w] for w in key_words])
sim_matrix = cosine_similarity(embeddings)

df_sim = pd.DataFrame(
    sim_matrix.round(3),
    index=key_words,
    columns=key_words
)
print(df_sim)
print("""
↑ REMEMBER in BoW one-hot, ALL off-diagonal values were 0.0.
  Now words have REAL similarity scores!
  Words that appear in similar contexts get similar vectors.

  NOTE: With only 6 tiny documents, these similarities are noisy.
  With millions of sentences, cat-dog would be ~0.7-0.8.
""")

# =============================================================
# --- STEP 4: MOST SIMILAR WORDS ---
# =============================================================
print("=== MOST SIMILAR WORDS (Word2Vec learned these relationships) ===\n")

for word in ["cat", "dog", "garden"]:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=5)
        print(f"Most similar to '{word}':")
        for sim_word, score in similar:
            print(f"  {sim_word:15s} → {score:.3f}")
        print()

# =============================================================
# --- STEP 5: DOCUMENT VECTORS (average word embeddings) ---
# =============================================================
print("=== DOCUMENT VECTORS (average of word embeddings) ===\n")

def doc_vector(tokens, model):
    """Average the word vectors for all tokens in a document."""
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.wv.vector_size)
    return np.mean(vecs, axis=0)

doc_vecs = np.array([doc_vector(tokens, model) for tokens in docs_tokenized])

print("Document vectors (10-dimensional):\n")
for i, vec in enumerate(doc_vecs):
    print(f"D{i+1}: {np.array2string(vec, precision=3, suppress_small=True)}")

# Document similarity using dense vectors
sim_docs = cosine_similarity(doc_vecs)
df_doc_sim = pd.DataFrame(
    sim_docs.round(3),
    index=[f"D{i+1}" for i in range(len(docs))],
    columns=[f"D{i+1}" for i in range(len(docs))]
)
print(f"\nDocument similarity (dense Word2Vec):\n")
print(df_doc_sim)

# =============================================================
# --- STEP 6: THE FULL PROGRESSION ---
# =============================================================
print("""
=== THE FULL PROGRESSION ===

1. BAG OF WORDS (one-hot)
   'cat' = [0, 0, 1, 0, 0, 0, 0, ...]     ← sparse, no meaning
   'dog' = [0, 0, 0, 0, 0, 1, 0, ...]     ← cosine(cat, dog) = 0.0
   Words are just IDs. No similarity possible.

2. TF-IDF (weighted one-hot)
   'cat' = [0, 0, 0.34, 0, 0, 0, 0, ...]  ← sparse, weighted by importance
   'dog' = [0, 0, 0, 0, 0, 0.52, 0, ...]  ← cosine(cat, dog) = STILL 0.0
   Better at ranking which words matter, but words still isolated.

3. WORD2VEC (dense embeddings)
   'cat' = [0.23, -0.11, 0.87, 0.44, ...]  ← dense, learned from context
   'dog' = [0.19, -0.08, 0.91, 0.38, ...]  ← cosine(cat, dog) ≈ 0.8!
   Words that appear in similar contexts → similar vectors.
   "You shall know a word by the company it keeps." (J.R. Firth, 1957)

HOW WORD2VEC LEARNS:
   Skip-gram asks: given 'cat', predict nearby words ('sat', 'mat', 'purred').
   Since 'dog' also appears near 'sat', 'the', 'and' →
   the model is forced to give 'cat' and 'dog' similar vectors
   because they need to predict similar neighbors.
""")

=== TOKENIZED DOCUMENTS ===

D1: ['the', 'cat', 'sat', 'on', 'the', 'mat', 'and', 'the', 'cat', 'purred', 'loudly']

D2: ['the', 'dog', 'sat', 'on', 'the', 'rug', 'and', 'barked', 'at', 'the', 'cat']

D3: ['a', 'bird', 'flew', 'over', 'the', 'mat', 'while', 'the', 'cat', 'watched', 'silently']

D4: ['the', 'dog', 'chased', 'the', 'bird', 'around', 'the', 'garden', 'fence']

D5: ['a', 'cat', 'and', 'a', 'dog', 'can', 'be', 'good', 'friends', 'despite', 'their', 'differences']

D6: ['the', 'garden', 'had', 'a', 'mat', 'near', 'the', 'fence', 'where', 'the', 'bird', 'nested']

=== WORD2VEC MODEL TRAINED ===
Vocabulary: 34 words
Embedding dimensions: 10

=== WORD EMBEDDINGS (each word is now a dense vector) ===

'cat       ' → [ 0.022  0.195 -0.068  0.502  0.732  0.132 -0.209 -0.316 -0.225 -0.091]
'dog       ' → [ 0.066  0.007 -0.129  0.348  0.537 -0.035 -0.155 -0.255 -0.18  -0.037]
'bird      ' → [-0.052  0.127 -0.059  0.326  0.448  0.153 -0.176 -0.106 -0.194 -0.004]
'mat       ' → [ 0.10